# Taller 1 — TEL328 — 2026

**Unidad 1: ruido, filtrado, segmentación, cuantización, detección de bordes y seguimiento**

**Publicación:** viernes 28 de agosto de 2026  
**Fecha límite:** domingo 6 de septiembre de 2026, 23:59  
**Modalidad:** individual  
**Entregable:** un archivo `.ipynb`

**Nombre:**  
**RUT:**  

---

## Objetivo

En este taller se trabajará con un problema de procesamiento digital de imágenes que requiere **analizar una imagen, seleccionar técnicas apropiadas, probar alternativas y justificar las decisiones tomadas**.

El objetivo no es implementar desde cero algoritmos que ya existen en librerías. Se espera que utilice correctamente las herramientas disponibles y que pueda **interpretar sus resultados, seleccionar parámetros y evaluar sus ventajas y limitaciones**.

Durante el taller se utilizarán contenidos vistos en la Unidad 1:

- manejo y modificación de píxeles;
- ruido;
- filtros de suavizado;
- segmentación mediante umbral;
- K-Means;
- Median Cut;
- Sobel;
- Canny;
- Mean-Shift;
- evaluación cuantitativa de resultados.


## 1. Reglas generales

1. El taller es **individual**.
2. Se permite utilizar documentación oficial de las librerías y material de clases.
3. Se permite utilizar herramientas de IA como apoyo, pero el estudiante debe comprender y poder explicar el código entregado.
4. No es necesario implementar algoritmos de procesamiento de imágenes desde cero cuando exista una implementación estándar en las librerías utilizadas.
5. Todas las decisiones relevantes deben estar acompañadas por una breve explicación en celdas Markdown.
6. Los resultados deben poder reproducirse ejecutando el notebook desde el comienzo.
7. No se aceptarán notebooks donde los resultados estén únicamente insertados como imágenes sin el código que los genera.
8. Si utiliza código externo, debe indicar claramente su procedencia.
9. El notebook debe ejecutarse sin errores utilizando un entorno estándar de Python con NumPy, Matplotlib, OpenCV y scikit-learn.
10. Se podrá solicitar una explicación breve de alguna parte del notebook durante la corrección.

### Entrega

El archivo debe llamarse:

`Taller1_Nombre_Apellido.ipynb`

La entrega vence el **domingo 6 de septiembre de 2026 a las 23:59**.

Se recomienda ejecutar **Kernel → Restart & Run All** antes de entregar para comprobar que el notebook completo funciona correctamente.


# 2. Preparación del entorno

Ejecute las siguientes celdas para importar las librerías necesarias.

Puede agregar otras librerías si las necesita, pero no utilice librerías que automaticen completamente el análisis solicitado sin explicar qué están haciendo.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.cluster import KMeans

print("OpenCV:", cv2.__version__)
print("NumPy:", np.__version__)


## 2.1. Reproducibilidad

Para que algunos parámetros del taller sean reproducibles, se utilizará una semilla derivada de su RUT.

Complete la siguiente celda. La semilla se utilizará únicamente para generar perturbaciones reproducibles durante el taller.


In [ ]:
# Escriba su RUT sin puntos ni guion.
# Ejemplo ficticio: 123456789
RUT = 0

SEED = int(str(RUT)[-6:]) if RUT != 0 else 123456
rng = np.random.default_rng(SEED)

print("Semilla utilizada:", SEED)


# 3. Datos del taller

Utilice las imágenes y el video proporcionados junto con este notebook.

Se trabajará principalmente con una imagen de referencia denominada:

`imagen_taller1.jpg`

y una secuencia de video denominada:

`video_taller1.mp4`

**No cambie las imágenes originales.**


In [ ]:
IMAGE_PATH = "imagen_taller1.jpg"
VIDEO_PATH = "video_taller1.mp4"

img_bgr = cv2.imread(IMAGE_PATH)

if img_bgr is None:
    raise FileNotFoundError(f"No se pudo cargar {IMAGE_PATH}")

img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

print("Dimensiones:", img_rgb.shape)
print("Tipo:", img_rgb.dtype)


In [ ]:
plt.figure(figsize=(9, 6))
plt.imshow(img_rgb)
plt.axis("off")
plt.title("Imagen de referencia")
plt.show()


# 4. Manejo de píxeles

Antes de utilizar algoritmos de alto nivel, trabajaremos directamente con los datos de la imagen.

Una imagen RGB puede representarse como un arreglo:

`alto × ancho × 3`

donde los tres canales corresponden a rojo, verde y azul.

## 4.1. Exploración

Realice lo siguiente:

1. Muestre las dimensiones de la imagen.
2. Obtenga el valor RGB de un píxel ubicado en una posición que usted elija.
3. Muestre los valores mínimo, máximo y promedio de cada canal.
4. Seleccione una pequeña región rectangular de la imagen y muestre sus valores.
5. Modifique una copia de la imagen pintando esa región con un color elegido por usted.

### Análisis

Explique brevemente:

- ¿Qué representa cada número almacenado en un píxel?
- ¿Qué diferencia existe entre modificar un canal y modificar los tres canales?
- ¿Qué ocurre si se modifica directamente un rango de píxeles fuera del intervalo `[0,255]`?


In [ ]:
# Complete esta sección.
# Seleccione usted las coordenadas del píxel y de la región.

y, x = 100, 100
print("Píxel RGB:", img_rgb[y, x])

for nombre, canal in zip(["R", "G", "B"], range(3)):
    datos = img_rgb[:, :, canal]
    print(nombre, "min:", datos.min(),
          "max:", datos.max(),
          "promedio:", datos.mean())


## 4.2. Conversión a escala de grises

Convierta la imagen a escala de grises utilizando OpenCV.

Luego implemente **una segunda conversión utilizando la fórmula de luminancia**:

$$
Y = 0.299R + 0.587G + 0.114B
$$

Compare ambas imágenes.

> No se espera que los resultados sean exactamente iguales debido a diferencias de implementación, redondeo y representación numérica.


In [ ]:
gray_cv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

# Implemente aquí la conversión mediante luminancia.
gray_manual = None

# Complete la implementación.


**Análisis:** compare ambas conversiones. ¿Dónde observa diferencias? ¿Por qué una conversión de escala de grises no consiste simplemente en calcular el promedio de R, G y B?


# 5. Ruido: diagnosticar antes de filtrar

En esta sección se generarán imágenes con diferentes tipos de ruido.

Se utilizarán dos modelos:

- **Ruido gaussiano:** pequeñas variaciones aleatorias alrededor del valor original de los píxeles.
- **Ruido sal y pimienta:** algunos píxeles son reemplazados por valores muy claros o muy oscuros.

La idea es observar que **el tipo de ruido influye directamente en la elección del filtro**.


In [ ]:
img_float = img_rgb.astype(np.float32) / 255.0

# Ruido gaussiano
sigma = 0.08
noise = rng.normal(0, sigma, img_float.shape)
noisy_gaussian = np.clip(img_float + noise, 0, 1)

# Ruido sal y pimienta
noisy_sp = img_float.copy()
prob = 0.04
mask = rng.random(img_float.shape[:2])

noisy_sp[mask < prob/2] = 0
noisy_sp[(mask >= prob/2) & (mask < prob)] = 1

noisy_gaussian = (noisy_gaussian * 255).astype(np.uint8)
noisy_sp = (noisy_sp * 255).astype(np.uint8)


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 5))

ax[0].imshow(img_rgb)
ax[0].set_title("Original")

ax[1].imshow(noisy_gaussian)
ax[1].set_title("Ruido gaussiano")

ax[2].imshow(noisy_sp)
ax[2].set_title("Ruido sal y pimienta")

for a in ax:
    a.axis("off")

plt.show()


### Preguntas

1. ¿Qué diferencias visuales permiten distinguir ambos tipos de ruido?
2. ¿Cuál de los dos espera que sea más afectado por un filtro de media? ¿Por qué?
3. ¿Cuál de los dos espera que sea más afectado por un filtro de mediana? ¿Por qué?

No se limite a indicar "se ve mejor": relacione su respuesta con el funcionamiento de cada filtro.


# 6. Selección de filtros

Ahora debe comprobar experimentalmente sus hipótesis.

Para cada tipo de ruido:

- pruebe al menos **tres filtros**;
- incluya filtros de media, gaussiano y mediana;
- pruebe al menos **dos tamaños de ventana**;
- calcule MAE, MSE y PSNR respecto de la imagen original.

Puede utilizar las implementaciones de OpenCV.

### Importante

No se pide probar todas las combinaciones posibles. Diseñe un experimento razonable que permita determinar qué filtro funciona mejor para cada tipo de ruido.


In [ ]:
def mae(a, b):
    return np.mean(np.abs(a.astype(np.float32) - b.astype(np.float32)))

def mse(a, b):
    return np.mean((a.astype(np.float32) - b.astype(np.float32)) ** 2)

def psnr(a, b):
    err = mse(a, b)
    if err == 0:
        return np.inf
    return 10 * np.log10((255.0 ** 2) / err)

# Complete el experimento.
# Construya una tabla con:
# ruido, filtro, tamaño, MAE, MSE y PSNR


### Análisis de resultados

A partir de sus resultados:

1. Determine el mejor filtro para el ruido gaussiano.
2. Determine el mejor filtro para el ruido sal y pimienta.
3. ¿Coincide el resultado cuantitativo con su evaluación visual?
4. ¿Qué ocurre al aumentar el tamaño de la ventana?
5. Explique un caso donde una métrica pueda sugerir un resultado "mejor", pero visualmente usted prefiera otro resultado.


# 7. Segmentación mediante umbral

La **segmentación** consiste en separar una imagen en regiones con diferentes características.

En esta sección se busca separar una región de interés del resto de la imagen.

Llamaremos **primer plano** al objeto o región que queremos conservar y **fondo** al resto de la imagen.

Pruebe:

1. un umbral global elegido manualmente;
2. Otsu;
3. umbral adaptativo.

No se le entrega el valor correcto del umbral: deberá determinarlo experimentalmente.


In [ ]:
# Trabaje con la imagen en escala de grises.
# Pruebe al menos las tres estrategias solicitadas.

# umbral manual
threshold_manual = None

# Otsu
mask_otsu = None

# Umbral adaptativo
mask_adapt = None


### Comparación

Muestre las tres máscaras y compare:

- regiones correctamente separadas;
- regiones incorrectamente clasificadas;
- sensibilidad a iluminación no uniforme;
- presencia de ruido o pequeños objetos.

**Pregunta:** ¿cuál de los tres métodos elegiría para este problema? Justifique utilizando evidencia obtenida de sus resultados.


# 8. Segmentación mediante K-Means

Ahora utilice **K-Means disponible en scikit-learn** para realizar una segmentación basada en color.

No debe implementar K-Means desde cero.

Realice experimentos con al menos:

- K = 3
- K = 5
- K = 8

Primero utilice solamente RGB como características.

Después, si lo considera pertinente, realice una segunda prueba utilizando otra representación de color disponible en OpenCV.

### Preguntas

1. ¿Qué regiones de la imagen quedan agrupadas juntas?
2. ¿Qué ocurre al aumentar K?
3. ¿Un valor mayor de K necesariamente produce una segmentación más útil?
4. ¿Qué ventajas y desventajas observa frente al umbral?


In [ ]:
pixels = img_rgb.reshape(-1, 3).astype(np.float32)

resultados_kmeans = {}

for k in [3, 5, 8]:
    # Complete el código.
    pass


# 9. Cuantización de color: K-Means vs. Median Cut

La **cuantización de color** consiste en representar una imagen utilizando una cantidad reducida de colores.

Utilice:

- K-Means;
- Median Cut mediante una implementación disponible en una librería apropiada o la implementación proporcionada por el profesor en las diapositivas de las clases.

No es necesario programar estos algoritmos desde cero.

Construya imágenes cuantizadas con:

- 4 colores;
- 8 colores;
- 16 colores.

### Compare

Para cada cantidad de colores:

- calidad visual;
- MSE;
- PSNR;
- tiempo de ejecución;
- preservación de detalles.

### Pregunta central

> Si tuviera que reducir una imagen a solamente 8 colores para una aplicación real, ¿elegiría K-Means o Median Cut? ¿Por qué?

Su respuesta debe basarse en sus experimentos, no solamente en una descripción teórica de los algoritmos.


In [ ]:
# Complete la comparación K-Means vs Median Cut.
# Registre los resultados en una tabla.


# 10. Detección de bordes: Sobel

Un borde corresponde a una zona donde existe un cambio significativo de intensidad.

Utilice Sobel para obtener:

- derivada horizontal;
- derivada vertical;
- magnitud del gradiente.

Realice también una implementación utilizando explícitamente los kernels de Sobel mediante una operación de convolución disponible en OpenCV.

No es necesario implementar manualmente la convolución píxel por píxel.


In [ ]:
# Complete:
# 1. Gradiente horizontal
# 2. Gradiente vertical
# 3. Magnitud


### Análisis

Muestre las tres imágenes y responda:

1. ¿Qué información aporta la derivada horizontal?
2. ¿Qué información aporta la derivada vertical?
3. ¿Qué representa la magnitud del gradiente?
4. ¿Qué ocurre con los bordes cuando aumenta el ruido?


# 11. Canny

Aplique el detector de bordes de Canny con al menos **tres configuraciones de umbrales**.

Canny utiliza varias etapas internas para obtener bordes más definidos. No es necesario implementar Canny desde cero.

Para cada configuración, muestre:

- imagen de bordes;
- cantidad de píxeles detectados como borde;
- una breve interpretación.

### Análisis

1. ¿Qué efecto tiene aumentar los umbrales?
2. ¿Qué configuración genera demasiados bordes falsos?
3. ¿Cuál considera más apropiada para esta imagen?
4. ¿Por qué Sobel y Canny pueden producir resultados diferentes aunque ambos detecten bordes?


In [ ]:
# Complete el experimento con Canny.
# Pruebe al menos tres pares de umbrales.


# 12. Seguimiento de un objeto mediante Mean-Shift

Ahora trabajaremos con una secuencia de video.

El objetivo es seguir un objeto a medida que cambia su posición.

**Mean-Shift** es un método que busca desplazar una ventana hacia la zona que presenta una mayor concentración de características similares a las del objeto que queremos seguir.

En este contexto:

- **ROI (región de interés):** rectángulo inicial que contiene el objeto.
- **Histograma:** representación de la distribución de colores del objeto.
- **Backprojection (retroproyección):** imagen que indica qué píxeles se parecen al color del objeto según su histograma.
- **Tracking (seguimiento):** proceso de actualizar la posición del objeto a través de los frames o cuadros del video.

Utilice las funciones disponibles en OpenCV. **No implemente Mean-Shift desde cero.**


In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise FileNotFoundError(f"No se pudo abrir {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS)
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print("FPS:", fps)
print("Frames:", n_frames)
print("Resolución:", width, "x", height)

cap.release()


## 12.1. Selección de la región inicial

Seleccione manualmente una ROI que contenga el objeto a seguir.

Registre:

- coordenada x;
- coordenada y;
- ancho;
- alto.

Explique brevemente por qué eligió esa región.

**Advertencia:** una ROI demasiado pequeña puede no representar adecuadamente el objeto; una ROI demasiado grande puede incorporar demasiado fondo.


In [ ]:
# Complete con la ROI inicial.
x, y, w, h = 0, 0, 0, 0


## 12.2. Histograma y retroproyección

Construya el histograma de color de la ROI y genere su retroproyección sobre los frames.

Muestre al menos:

1. ROI inicial;
2. histograma;
3. retroproyección de un frame.

Explique qué zonas de la retroproyección deberían presentar valores altos y por qué.


In [ ]:
# Complete la construcción del histograma y la retroproyección.


## 12.3. Tracking

Realice el seguimiento mediante Mean-Shift.

Muestre el video o una selección representativa de frames con el rectángulo de seguimiento.

Además, registre para cada frame:

- posición x;
- posición y;
- ancho;
- alto.

Genere posteriormente una gráfica de la trayectoria del centro de la ventana.


In [ ]:
# Complete el tracking mediante Mean-Shift.
# Guarde la posición estimada para cada frame.


## 12.4. Evaluación del tracking

Para evaluar el comportamiento del tracker, seleccione manualmente al menos **10 frames distribuidos a lo largo del video** y determine aproximadamente la posición real del objeto.

Compare la posición real con la posición entregada por Mean-Shift.

Calcule un error de posición usando:

$$
e = \sqrt{(x_r-x_e)^2 + (y_r-y_e)^2}
$$

donde:

- $(x_r,y_r)$ corresponde a la posición real;
- $(x_e,y_e)$ corresponde a la posición estimada.

### Análisis

1. ¿En qué frame se produce el mayor error?
2. ¿Qué característica de la imagen puede explicar ese error?
3. ¿El objeto cambia de tamaño?
4. ¿Cambia su iluminación o color?
5. ¿Aparecen objetos similares?
6. ¿Qué modificación haría para mejorar el tracking?


# 13. Desafío integrador

En esta última parte deberá diseñar **su propio pipeline de procesamiento**.

Un pipeline es una secuencia ordenada de operaciones donde la salida de una etapa se utiliza como entrada de la siguiente.

Debe utilizar **al menos cuatro técnicas diferentes vistas en la Unidad 1**.

Por ejemplo:

`imagen → reducción de ruido → segmentación → detección de bordes → análisis`

El ejemplo anterior es solamente ilustrativo. **No debe copiar necesariamente esa secuencia.**

## Requisitos

Su propuesta debe:

1. plantear claramente un objetivo;
2. utilizar al menos cuatro técnicas de la unidad;
3. justificar el orden elegido;
4. mostrar el resultado después de cada etapa importante;
5. incluir al menos una alternativa que haya probado y descartado;
6. incluir una evaluación cuantitativa cuando sea posible;
7. discutir al menos dos limitaciones de su solución.

### Pregunta final

> Si tuviera que convertir este pipeline en una aplicación real, ¿qué parte modificaría primero y por qué?


In [ ]:
# Desarrolle aquí su pipeline.
# Puede agregar tantas celdas como necesite.


# 14. Conclusiones

Escriba una conclusión de **al menos 300 palabras**.

Debe responder, como mínimo:

1. ¿Qué técnica le resultó más útil y por qué?
2. ¿Qué técnica produjo resultados menos satisfactorios?
3. ¿Qué aprendió sobre la relación entre parámetros y resultados?
4. ¿Qué diferencia existe entre obtener un resultado visualmente atractivo y obtener un resultado cuantitativamente bueno?
5. ¿Qué aprendió sobre la selección de técnicas para resolver un problema de procesamiento de imágenes?

No se aceptará una conclusión que se limite a enumerar las funciones utilizadas.


# 15. Checklist antes de entregar

Antes de entregar compruebe:

- [ ] Mi nombre y RUT están indicados.
- [ ] El notebook se ejecuta desde el comienzo sin errores.
- [ ] Todas las figuras tienen títulos o etiquetas comprensibles.
- [ ] Todas las preguntas están respondidas.
- [ ] Los análisis están escritos en Markdown.
- [ ] Los resultados fueron generados mediante código.
- [ ] Incluí tablas o métricas cuando fueron solicitadas.
- [ ] Mi pipeline integrador utiliza al menos cuatro técnicas.
- [ ] Expliqué mis decisiones y no solamente mostré resultados.
- [ ] Ejecuté **Restart & Run All** antes de entregar.
